# 🎙️ 어쩌다지식 - GPT-SoVITS 음성 합성
Google Colab에서 실행. 내 목소리로 클로닝된 나레이션을 생성합니다.

In [ ]:
# ==============================
# 설정 (여기만 수정하세요)
# ==============================
RUN_ID = "여기에_run_id_입력"          # main.py 실행 후 출력된 run_id
DRIVE_BASE = "/content/drive/MyDrive/어쩌다지식"
VOICE_SAMPLE_PATH = "/content/drive/MyDrive/어쩌다지식/voice_sample.wav"  # 본인 목소리 샘플
USE_PRETRAINED = True   # True: 학습된 모델 사용 / False: 처음 학습
MODEL_PATH = "/content/drive/MyDrive/어쩌다지식/gpt_sovits_model"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Install GPT-SoVITS
!git clone https://github.com/RVC-Boss/GPT-SoVITS.git
%cd GPT-SoVITS
!pip install -r requirements.txt -q

In [ ]:
import json
from pathlib import Path

run_state_path = Path(f"{DRIVE_BASE}/runs/{RUN_ID}.json")
assert run_state_path.exists(), f"Run state not found: {run_state_path}"

with open(run_state_path) as f:
    state = json.load(f)

script = state["script"]
scenes = script["scenes"]
print(f"\u2705 Run: {RUN_ID}")
print(f"\U0001f4dd \uc81c\ubaa9: {script['title']}")
print(f"\U0001f3ac \uc528 \uc218: {len(scenes)}\uac1c")

In [ ]:
import os

if USE_PRETRAINED and Path(MODEL_PATH).exists():
    print("\u2705 \ud559\uc2b5\ub41c \ubaa8\ub378 \ub85c\ub4dc \uc911...")
    # Load existing model - path will be used in synthesis
    GPT_MODEL = f"{MODEL_PATH}/gpt_weights.ckpt"
    SOVITS_MODEL = f"{MODEL_PATH}/sovits_weights.pth"
    REF_AUDIO = VOICE_SAMPLE_PATH
    print(f"GPT: {GPT_MODEL}")
    print(f"SoVITS: {SOVITS_MODEL}")
else:
    print("\U0001f393 \ubaa9\uc18c\ub9ac \ud559\uc2b5 \uc2dc\uc791...")
    print(f"\uc0d8\ud50c \ud30c\uc77c: {VOICE_SAMPLE_PATH}")
    # Training commands for GPT-SoVITS
    # Users need to have their voice sample at VOICE_SAMPLE_PATH
    !python GPT_SoVITS/prepare_datasets/asr.py \
        --voice_sample {VOICE_SAMPLE_PATH} \
        --output_dir /content/training_data
    print("\u26a0\ufe0f \ud559\uc2b5\uc740 GPT-SoVITS \uacf5\uc2dd \uac00\uc774\ub4dc\ub97c \ucc38\uc870\ud558\uc138\uc694")
    print("\ud559\uc2b5 \uc644\ub8cc \ud6c4 \ubaa8\ub378\uc744 Drive\uc5d0 \uc800\uc7a5\ud558\uace0 USE_PRETRAINED=True\ub85c \ubcc0\uacbd\ud558\uc138\uc694")

In [ ]:
import sys
sys.path.insert(0, '/content/GPT-SoVITS')

from GPT_SoVITS.inference_webui import get_tts_wav  # may vary by version
import soundfile as sf
from pathlib import Path

output_audio_dir = Path(f"{DRIVE_BASE}/runs/{RUN_ID}/audio")
output_audio_dir.mkdir(parents=True, exist_ok=True)

EMOTION_SPEED_MAP = {
    "hook":       0.90,
    "surprising": 0.88,
    "curious":    1.00,
    "calm":       1.05,
    "building":   0.95,
    "warm":       1.05,
    "cta":        1.00,
    "neutral":    1.00,
}

print(f"\U0001f399\ufe0f {len(scenes)}\uac1c \uc528 \uc74c\uc131 \ud569\uc131 \uc2dc\uc791...")
for i, scene in enumerate(scenes):
    out_path = output_audio_dir / f"scene_{i:03d}.mp3"
    if out_path.exists():
        print(f"  \uc528 {i}: \uce90\uc2dc \uc0ac\uc6a9 ({out_path.name})")
        continue
    
    emotion = scene.get("emotion", "neutral")
    speed = EMOTION_SPEED_MAP.get(emotion, 1.0)
    text = scene["narration"]
    
    print(f"  \uc528 {i}/{len(scenes)-1}: [{emotion}] {text[:40]}...")
    
    # GPT-SoVITS inference
    # Note: exact API depends on GPT-SoVITS version
    # This is the standard inference call pattern
    synthesis_result = get_tts_wav(
        ref_wav_path=REF_AUDIO,
        prompt_text="",  # reference text (can be empty)
        prompt_language="ko",
        text=text,
        text_language="ko",
        how_to_cut="\u6309\u53e5\u53f7\u5207",
        top_k=15,
        top_p=1.0,
        temperature=1.0,
        ref_free=False,
        speed=speed,
    )
    
    # Save as wav first then convert to mp3
    wav_path = output_audio_dir / f"scene_{i:03d}.wav"
    for sr, audio_data in synthesis_result:
        sf.write(str(wav_path), audio_data, sr)
        break
    
    # Convert to mp3
    import subprocess
    subprocess.run(["ffmpeg", "-y", "-i", str(wav_path), str(out_path)], 
                   capture_output=True)
    wav_path.unlink(missing_ok=True)
    print(f"    \u2192 {out_path.name} \u2705")

print(f"\n\u2705 \uc74c\uc131 \ud569\uc131 \uc644\ub8cc! Drive\uc5d0 \uc800\uc7a5\ub428: {output_audio_dir}")

In [ ]:
# Mark audio files in run state
audio_files = sorted(output_audio_dir.glob("scene_*.mp3"))
state["audio_files"] = [str(p) for p in audio_files]
state["tts_provider"] = "gpt_sovits_colab"
state.setdefault("stages", {})["tts"] = {
    "status": "completed",
    "provider": "gpt_sovits_colab",
    "file_count": len(audio_files),
}

with open(run_state_path, "w") as f:
    json.dump(state, f, ensure_ascii=False, indent=2)

print(f"\u2705 \uc0c1\ud0dc \uc5c5\ub370\uc774\ud2b8 \uc644\ub8cc")
print(f"\U0001f3b5 \uc0dd\uc131\ub41c \ud30c\uc77c: {len(audio_files)}\uac1c")
print(f"\n\ub2e4\uc74c \ub2e8\uacc4:")
print(f"  1. Notebook 02 (Wan2.1 \uc601\uc0c1) \uc2e4\ud589")
print(f"  2. \ub610\ub294 \ubc14\ub85c \uc870\ud569: python pipeline/main.py --skip-to assemble --run-id {RUN_ID}")